# OCR Evaluation on Weave — Model Comparison

Runs a [Weave Evaluation](https://weave-docs.wandb.ai/guides/core-types/evaluations) over the published `ocr-eval` dataset, comparing vision models with the **prompt held constant**:

- **Prompt**: the exact OCR prompt used by [jbutt-wandb/ocr_handwritten_notes](https://github.com/jbutt-wandb/ocr_handwritten_notes) (`build_ocr_prompt(contains_latex=True, contains_diagrams=True)`), vendored verbatim from `backend/prompts/ocr_prompts.py`
- **Models**: the app's registry of vision models on [W&B Inference](https://docs.wandb.ai/guides/inference/) — Kimi K2.6 (app default), Gemma 4 31B, Qwen 3.6 35B A3B
- **Predict path**: faithful to the app's `OCRModel.predict` — 1600 px downscale, strict JSON-schema response format, `clean_markdown` post-processing
- **Scoring**: [jiwer](https://jitsi.github.io/jiwer/) OCR metrics — WER, MER, WIL, WIP, CER — on both **raw** text and a **normalized** view (lowercase, punctuation stripped, whitespace collapsed)

In [ ]:
import base64
import io
import json
import logging
import os
import re

import jiwer
import pandas as pd
import weave
from dotenv import load_dotenv
from openai import AsyncOpenAI
from PIL import Image

logging.getLogger("weave").setLevel(logging.ERROR)

load_dotenv()
ENTITY = os.environ["ENTITY"]
PROJECT = os.environ["PROJECT"]

client = weave.init(f"{ENTITY}/{PROJECT}")
print(f"Weave project: {ENTITY}/{PROJECT}")

## Load the eval dataset

In [ ]:
dataset = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/ocr-eval:latest").get()
print(f"Loaded dataset '{dataset.name}' with {len(dataset.rows)} rows")

In [ ]:
dataset_df = dataset.to_pandas()
dataset_df.head()

## OCR model

Faithful to the app's `OCRModel.predict` (`backend/services/models.py`): prompt sent as a user text part alongside the image, images downscaled to max **1600 px** (W&B Inference rejects very large payloads), strict JSON-schema response format (`{"markdown": ...}`) with raw-content fallback, `max_tokens=32768` (Gemma/Qwen spend heavily on reasoning before answering), and the app's `clean_markdown` post-processing. `weave.Model` versions `model_name` and `prompt`, so the compare view shows the model as the only attribute that differs between runs.

In [ ]:
# App constants (backend/services/inference_service.py)
MAX_IMAGE_DIMENSION = 1600
MAX_OUTPUT_TOKENS = 32_768

OCR_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "OCROutput",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {"markdown": {"type": "string"}},
            "required": ["markdown"],
            "additionalProperties": False,
        },
    },
}

In [ ]:
inference_client = AsyncOpenAI(
    base_url="https://api.inference.wandb.ai/v1",
    api_key=os.environ["WANDB_API_KEY"],
    default_headers={"OpenAI-Project": f"{ENTITY}/{PROJECT}"},
)


def image_to_data_uri(image: Image.Image, max_side: int = MAX_IMAGE_DIMENSION, quality: int = 85) -> str:
    img = image.copy()
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=quality)
    return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"


def clean_markdown(text: str) -> str:
    """Remove any markdown code fences that wrap the entire content (app's post-processing)."""
    text = re.sub(r'^```(?:markdown)?\s*\n?', '', text.strip())
    text = re.sub(r'\n?```\s*$', '', text.strip())
    return text.strip()


In [ ]:
class OCRModel(weave.Model):
    model_name: str
    prompt: str

    @weave.op
    async def predict(self, image: Image.Image) -> str:
        response = await inference_client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": self.prompt},
                        {"type": "image_url", "image_url": {"url": image_to_data_uri(image)}},
                    ],
                }
            ],
            response_format=OCR_RESPONSE_FORMAT,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        content = (response.choices[0].message.content or "").strip()

        try:
            markdown = json.loads(content)["markdown"]
        except Exception:
            markdown = content
        return clean_markdown(markdown)

## The constant prompt

Vendored verbatim from the app's [`backend/prompts/ocr_prompts.py`](https://github.com/jbutt-wandb/ocr_handwritten_notes/blob/main/backend/prompts/ocr_prompts.py). The eval uses `build_ocr_prompt(contains_latex=True, contains_diagrams=True)` — the fullest prompt the app can build — held constant across all models.

In [ ]:
# Assemble Prompt - the same would be used in the backend service for inference

SYSTEM_PROMPT = """You are an expert OCR assistant specializing in converting handwritten notes to clean, well-formatted Markdown. Your task is to:

1. Accurately transcribe all handwritten text
2. Preserve the logical structure (headings, lists, paragraphs)
3. Maintain the original organization and flow of ideas
4. Use appropriate Markdown formatting (headers, bold, italics, lists)

Output ONLY the Markdown content, no explanations or commentary."""

LATEX_INSTRUCTIONS = """MATHEMATICAL EQUATIONS:
- Convert ALL mathematical expressions to LaTeX format
- Use inline math with single dollar signs: $x^2 + y^2 = r^2$
- Use display math with double dollar signs for standalone equations:
  $$\\int_{a}^{b} f(x) dx = F(b) - F(a)$$
- Properly escape special characters
- Use standard LaTeX notation:
  - Fractions: \\frac{numerator}{denominator}
  - Square roots: \\sqrt{x} or \\sqrt[n]{x}
  - Subscripts/superscripts: x_i, x^2
  - Greek letters: \\alpha, \\beta, \\gamma
  - Summations: \\sum_{i=1}^{n}
  - Integrals: \\int, \\iint, \\oint
  - Limits: \\lim_{x \\to \\infty}
- Preserve equation numbering if visible"""

DIAGRAM_INSTRUCTIONS = """GRAPHS AND DIAGRAMS:
- Describe diagrams in detail within markdown blockquotes or as descriptive text
- For graphs, describe:
  - Type of graph (line, bar, scatter, etc.)
  - Axes labels and scales
  - Key data points or trends
  - Any annotations or labels
- For diagrams, describe:
  - The overall structure and layout
  - Components and their relationships
  - Labels and text within the diagram
  - Arrows, connections, or flow directions
- Format descriptions clearly:

  > **Diagram: [Title/Description]**
  > - Component A connects to Component B
  > - Arrow indicates flow direction from X to Y

- If the diagram contains mathematical relationships, express them in LaTeX"""

OUTPUT_INSTRUCTIONS = """OUTPUT FORMAT:
- Return clean, properly formatted Markdown
- Use appropriate heading levels (# ## ###)
- Preserve bullet points and numbered lists
- Separate distinct sections with blank lines
- Do not include any preamble or explanation
- Start directly with the transcribed content
- Do NOT wrap output in code fences (no ```markdown```)
- Do NOT add page dividers or separators
- Output raw markdown text only"""


def build_ocr_prompt(
    contains_latex: bool,
    contains_diagrams: bool,
    custom_instructions: str = "",
) -> str:
    """Build a dynamic OCR prompt based on user options."""
    prompt_parts = [SYSTEM_PROMPT]
    if contains_latex:
        prompt_parts.append(LATEX_INSTRUCTIONS)
    if contains_diagrams:
        prompt_parts.append(DIAGRAM_INSTRUCTIONS)
    prompt_parts.append(OUTPUT_INSTRUCTIONS)
    if custom_instructions and custom_instructions.strip():
        prompt_parts.append(f"ADDITIONAL USER INSTRUCTIONS:\n{custom_instructions.strip()}")
    return "\n\n".join(prompt_parts)


# --- End vendored code ---

OCR_PROMPT = build_ocr_prompt(contains_latex=True, contains_diagrams=True)

# The app's model registry (backend/services/inference_service.py::AVAILABLE_MODELS)
MODELS = {
    "moonshotai/Kimi-K2.6": "kimi-k2.6",
    "google/gemma-4-31B-it": "gemma-4-31b",
    "Qwen/Qwen3.6-35B-A3B": "qwen3.6-35b",
    "DeepSeek V4.1-Flash": "deepseek-ai/DeepSeek-V4.1-Flash",
}

print(f"Prompt: {len(OCR_PROMPT)} chars | Models: {list(MODELS.values())}")

## jiwer scorer

Returns every metric twice: `raw` (exact text, LaTeX/markdown included) and `normalized` (lowercase, punctuation stripped, whitespace collapsed — content accuracy without formatting noise). Weave averages the numeric leaves across rows in the eval summary.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def jiwer_metrics(reference: str, hypothesis: str) -> dict:

    if not hypothesis.strip():
        return {"wer": 1.0, "mer": 1.0, "wil": 1.0, "wip": 0.0, "cer": 1.0}

    words = jiwer.process_words(reference, hypothesis)
    chars = jiwer.process_characters(reference, hypothesis)

    return {
        "wer": words.wer,
        "mer": words.mer,
        "wil": words.wil,
        "wip": words.wip,
        "cer": chars.cer,
    }


@weave.op
def ocr_metrics(ground_truth_text: str, output: str) -> dict:
    return {
        "raw": jiwer_metrics(ground_truth_text, output),
        "normalized": jiwer_metrics(normalize_text(ground_truth_text), normalize_text(output)),
    }

## Run the model comparison

One `Evaluation` (same dataset, same scorer), evaluated once per model with the prompt held constant — M models × N images = MN inference calls.

In [ ]:
evaluation = weave.Evaluation(
    name="ocr-model-comparison", 
    dataset=dataset, 
    scorers=[ocr_metrics]
    )

In [ ]:
MODELS

In [ ]:
# Run evaluation for each model and store results
results = {}

for model_id, slug in MODELS.items():

    model = OCRModel(
        name=f"ocr-{slug}",
        model_name=model_id,
        prompt=OCR_PROMPT,
    )

    print(f"Evaluating model: {model_id} ...")

    results[model_id] = await evaluation.evaluate(
        model,
        __weave={"display_name": f"ocr-{slug}"}
    )

print("\nAll eval runs complete.")
print(f"Compare them at: https://wandb.ai/{ENTITY}/{PROJECT}/weave/evaluations")

## Model leaderboard

Mean metrics per model, sorted by normalized WER (lower is better for WER/MER/WIL/CER; higher is better for WIP).

In [ ]:
import weave
from weave.flow import leaderboard
from weave.trace.ref_util import get_ref

# Pull evaluation results into a Weave table for further analysis
evaluations = client.get_evaluations()

# Check evaluation names
for eval in evaluations:
    print(eval.name)

filtered_eval = [i for i in evaluations if i.name == 'ocr-model-comparison'][0]

# Define the leaderboard
spec = leaderboard.Leaderboard(
    name="OCR Model Comparison Leaderboard",
    description="Comparing models on the OCR evaluation dataset using WER, MER, WIL, WIP, and CER metrics.",
    columns=[
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.cer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.mer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wil.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wip.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.cer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.mer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wil.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wip.mean",          # e.g. "mean", "true_fraction"
        ),
    ]
)

# Publish the leaderboard
weave.publish(spec)

# Optionally retrieve and print results
results = leaderboard.get_leaderboard_results(spec, client)